In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

tr = pd.read_csv("../data/processed/split_train.csv")
tr["text"] = tr["text"].fillna("")

tfidf = TfidfVectorizer(max_features=10000, stop_words="english", min_df=3)
X = tfidf.fit_transform(tr["text"])

svd = TruncatedSVD(n_components=100, random_state=42)
Z = normalize(svd.fit_transform(X))
print(X.shape, "->", Z.shape)

(26203, 10000) -> (26203, 100)


In [2]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

rng = np.random.RandomState(42)
sample = rng.choice(len(Z), 5000, replace=False)

for k in [5, 10, 15, 20, 30]:
    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=3, batch_size=2048)
    labels = km.fit_predict(Z)
    sil = silhouette_score(Z[sample], labels[sample])
    print(f"k={k:2d}  inertia={km.inertia_:8.1f}  silhouette={sil:.3f}")

k= 5  inertia= 23259.8  silhouette=0.037
k=10  inertia= 21582.5  silhouette=0.061
k=15  inertia= 20287.7  silhouette=0.082
k=20  inertia= 19682.5  silhouette=0.089
k=30  inertia= 18022.8  silhouette=0.108


In [3]:
k = 15   # change this to your choice
km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=3, batch_size=2048)
tr["cluster"] = km.fit_predict(Z)

terms = np.array(tfidf.get_feature_names_out())
centers = svd.inverse_transform(km.cluster_centers_)

for c in range(k):
    top = terms[np.argsort(centers[c])[-8:][::-1]]
    print(c, int((tr["cluster"] == c).sum()), "|", ", ".join(top))

0 2135 | size, inches, white, black, wall, 12, inch, color
1 2065 | scale, model, kit, car, racing, wheels, redcat, 10
2 1249 | building, lego, set, pieces, learning, kit, blocks, educational
3 1851 | set, swing, includes, twin, piece, table, wooden, comforter
4 1046 | puzzle, jigsaw, piece, pieces, 1000, great, condition, puzzles
5 5851 | kit, art, unique, box, includes, kids, fun, easy
6 1055 | plush, stuffed, animal, soft, toy, wild, aurora, bear
7 1316 | game, board, card, games, players, family, dice, ages
8 1398 | costume, rubie, child, dress, deluxe, mask, medium, includes
9 1769 | toy, baby, toys, ages, activity, fun, world, kids
10 851 | doll, barbie, baby, dolls, accessories, inch, hair, girl
11 1493 | condition, great, skate, inline, model, k2, scale, costume
12 1422 | party, pack, paper, amscan, birthday, supplies, 12, count
13 1493 | figure, action, scale, wwe, multicolor, inch, funko, series
14 1209 | play, kids, pretend, set, kitchen, fun, toy, playset


In [4]:
pd.crosstab(tr["cluster"], tr["main_category"]).loc[:, lambda t: t.sum() > 300]

main_category,"Arts, Crafts & Sewing",Baby Products,"Clothing, Shoes & Jewelry",Home & Kitchen,Sports & Outdoors,Toys & Games
cluster,,,,,,
0,23,280,139,561,269,760
1,0,0,10,20,161,1586
2,1,0,0,16,0,1143
3,6,14,8,563,64,1055
4,0,0,0,1,4,1035
5,109,47,205,347,677,3772
6,13,9,8,51,5,931
7,0,0,0,0,49,1195
8,0,0,812,1,5,461
